# Unidade 3 - Bloco prático da Aula 02: o campeonato de recursos

Compara RandomizedSearchCV (todos os candidatos com os dados completos) e HalvingRandomSearchCV (funil com fator 3). Acompanhe o funil de eliminação rodada a rodada e o tempo total de cada estratégia.

In [ ]:
import time
import numpy as np
from sklearn.datasets import make_classification
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import (HalvingRandomSearchCV,
                                     RandomizedSearchCV, StratifiedKFold,
                                     train_test_split)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from scipy.stats import randint

X, y = make_classification(n_samples=12000, n_features=20, n_informative=8,
                           weights=[0.8, 0.2], flip_y=0.02, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
dist = {"n_estimators": randint(50, 300),
        "max_depth": [4, 6, 8, 12, None],
        "min_samples_leaf": randint(1, 15),
        "max_features": ["sqrt", "log2", None]}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

rs = RandomizedSearchCV(rf, dist, n_iter=40, cv=cv, scoring="f1",
                        random_state=42, n_jobs=-1)
t0 = time.time(); rs.fit(X_tr, y_tr); t_rs = time.time() - t0

hs = HalvingRandomSearchCV(rf, dist, n_candidates=40,
                           resource="n_samples", min_resources=1000,
                           factor=3, cv=cv, scoring="f1",
                           random_state=42, n_jobs=-1)
t0 = time.time(); hs.fit(X_tr, y_tr); t_hs = time.time() - t0

print(f"Random  (40 candidatos, dados completos): "
      f"CV={rs.best_score_:.4f} teste={f1_score(y_te, rs.predict(X_te)):.4f} "
      f"tempo={t_rs:.0f}s")
print(f"Halving (40 candidatos, fator 3):         "
      f"CV={hs.best_score_:.4f} teste={f1_score(y_te, hs.predict(X_te)):.4f} "
      f"tempo={t_hs:.0f}s")